# osu!mania Difficulty Model - Colab Training

Use this when local training is too slow. In VS Code, click the kernel picker and choose `Colab` -> `New Colab Server`, then pick a GPU runtime and run top to bottom.


In [ ]:
# Setup
REPO_URL = "https://github.com/fatelvx/ODP-model.git"
BRANCH = ""  # Optional. Leave empty to use the repo default branch.

import os
import subprocess
from pathlib import Path

if not REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repo URL first.")

!rm -rf /content/mania-difficulty-model
clone_cmd = ["git", "clone", "--depth", "1"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [REPO_URL, "/content/mania-difficulty-model"]
subprocess.run(clone_cmd, check=True)
%cd /content/mania-difficulty-model
!pip install -e .


In [ ]:
# Check GPU and configure runtime-sensitive training options.
import subprocess
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
TRAIN_DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"
LOADER_WORKERS = 2 if CUDA_AVAILABLE else 0
AMP_MODE = "auto"

print("torch", torch.__version__)
print("cuda", CUDA_AVAILABLE)
print("train_device", TRAIN_DEVICE)
print("loader_workers", LOADER_WORKERS)
print("amp", AMP_MODE)
if CUDA_AVAILABLE:
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("WARNING: No GPU runtime detected. In Colab, choose Runtime > Change runtime type > GPU before real neural training.")


## Smoke Test

Run this first. It proves the training/report pipeline works before spending API calls or GPU time on real data.


In [ ]:
!python -m mania_difficulty.tools.make_synthetic_dataset --maps 128 --out data/processed/synthetic_colab
!python -m mania_difficulty.train \
  --labels data/processed/synthetic_colab/labels.csv \
  --sequences data/processed/synthetic_colab/sequences \
  --epochs 12 \
  --batch-size 32 \
  --grad-accum-steps 1 \
  --grad-clip-norm 1.0 \
  --checkpoint-metric val_mean_mae \
  --run-name colab_synthetic \
  --model lstm \
  --loader-workers {LOADER_WORKERS} \
  --device {TRAIN_DEVICE} \
  --amp {AMP_MODE}


In [ ]:
from IPython.display import Image, display, HTML

run_dir = Path("outputs/runs/colab_synthetic")
display(Image(str(run_dir / "learning_curve.png")))
display(Image(str(run_dir / "prediction_scatter.png")))
display(Image(str(run_dir / "prediction_errors.png")))
display(HTML((run_dir / "run_report.html").read_text(encoding="utf-8")))


## Real osu! Data

Use this after the smoke test. The credentials live only in this Colab session.


In [ ]:
import getpass
os.environ["OSU_CLIENT_ID"] = getpass.getpass("OSU_CLIENT_ID: ")
os.environ["OSU_CLIENT_SECRET"] = getpass.getpass("OSU_CLIENT_SECRET: ")


In [ ]:
# Start smaller than 2000 while debugging. Raise --target after the pipeline is stable.
!python -m mania_difficulty.data.fetch_maps --target 300 --out data/raw/beatmaps.csv
!python -m mania_difficulty.data.fetch_osu_files --maps data/raw/beatmaps.csv --out-dir data/raw/osu
!python -m mania_difficulty.data.parse_notes --maps data/raw/beatmaps.csv --osu-dir data/raw/osu --out-dir data/processed/sequences
!python -m mania_difficulty.data.fetch_scores --maps data/raw/beatmaps.csv --out data/processed/labels.csv --min-scores 30


In [ ]:
from pathlib import Path
from IPython.display import HTML, Image, display

!python -m mania_difficulty.tools.audit_dataset \
  --labels data/processed/labels.csv \
  --sequences data/processed/sequences \
  --out-dir outputs/dataset_audit_top100

audit_dir = Path("outputs/dataset_audit_top100")
display(HTML((audit_dir / "dataset_audit.html").read_text(encoding="utf-8")))
display(Image(str(audit_dir / "dataset_distributions.png")))


In [ ]:
import json
from pathlib import Path

!python -m mania_difficulty.tools.sweep_forest \
  --labels data/processed/labels.csv \
  --sequences data/processed/sequences \
  --out-dir outputs/forest_sweep_top100 \
  --cv-folds 5 \
  --group-column beatmapset_id \
  --trees 200,500 \
  --min-samples-leaf 1,2,4 \
  --max-features sqrt,0.75 \
  --feature-sets core,burst \
  --selection-metric mean_pairwise_order_accuracy \
  --workers -1

best_params = json.loads(Path("outputs/forest_sweep_top100/best_params.json").read_text(encoding="utf-8"))
FOREST_TREES = int(best_params["forest_trees"])
FOREST_MIN_SAMPLES_LEAF = int(best_params["forest_min_samples_leaf"])
FOREST_MAX_FEATURES = str(best_params["forest_max_features"])
FEATURE_SET = str(best_params.get("feature_set", "core"))
best_params


In [ ]:
!python -m mania_difficulty.train \
  --labels data/processed/labels.csv \
  --sequences data/processed/sequences \
  --run-name colab_forest_top100 \
  --model tabular_forest \
  --forest-trees {FOREST_TREES} \
  --forest-min-samples-leaf {FOREST_MIN_SAMPLES_LEAF} \
  --forest-max-features {FOREST_MAX_FEATURES} \
  --feature-set {FEATURE_SET} \
  --cv-folds 5 \
  --group-column beatmapset_id \
  --workers -1


In [ ]:
import json
import subprocess
from pathlib import Path

RUN_NEURAL_SWEEP = True
NEURAL_SWEEP_MAX_CANDIDATES = 4
GRAD_ACCUM_STEPS = 1  # Raise to 2 or 4 if the GPU runs out of memory.
GRAD_CLIP_NORM = 1.0  # Set 0 to disable gradient clipping.
CHECKPOINT_METRIC = "val_mean_mae"  # Use "val_mean_pairwise_order_accuracy" when rank direction matters most.
SAMPLE_WEIGHT_COLUMN = "score_count"  # Set to "" to disable label reliability weighting.
SAMPLE_WEIGHT_MIN = 0.25
SAMPLE_WEIGHT_MAX_VALUE = 100
RESUME_FINAL_TRAINING = True  # Re-running the final train cell resumes from last_checkpoint.pt.
USE_DRIVE_CHECKPOINT_BACKUP = False  # Set True to persist final neural checkpoints across runtime resets.
CHECKPOINT_BACKUP_DIR = "/content/drive/MyDrive/osu_mania_difficulty_checkpoints"

if USE_DRIVE_CHECKPOINT_BACKUP:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CHECKPOINT_BACKUP_DIR).mkdir(parents=True, exist_ok=True)

if RUN_NEURAL_SWEEP:
    subprocess.run(
        [
            "python",
            "-m",
            "mania_difficulty.tools.sweep_neural",
            "--labels",
            "data/processed/labels.csv",
            "--sequences",
            "data/processed/sequences",
            "--out-dir",
            "outputs/neural_sweep_top100",
            "--run-prefix",
            "neural_sweep_top100",
            "--models",
            "lstm",
            "--epochs",
            "30",
            "--patience",
            "6",
            "--batch-size",
            "32",
            "--grad-accum-steps",
            str(GRAD_ACCUM_STEPS),
            "--grad-clip-norm",
            str(GRAD_CLIP_NORM),
            "--checkpoint-metric",
            CHECKPOINT_METRIC,
            "--sample-weight-column",
            SAMPLE_WEIGHT_COLUMN,
            "--sample-weight-min",
            str(SAMPLE_WEIGHT_MIN),
            "--sample-weight-max-value",
            str(SAMPLE_WEIGHT_MAX_VALUE),
            "--lrs",
            "0.001,0.0005",
            "--weight-decays",
            "0.0001",
            "--lstm-embed-dims",
            "32,64",
            "--lstm-hidden-dims",
            "64,128",
            "--lstm-layers",
            "1,2",
            "--lstm-dropouts",
            "0.1,0.2",
            "--lstm-head-dropouts",
            "0.2",
            "--selection-metric",
            "mean_pairwise_order_accuracy",
            "--loader-workers",
            str(LOADER_WORKERS),
            "--max-candidates",
            str(NEURAL_SWEEP_MAX_CANDIDATES),
            "--group-column",
            "beatmapset_id",
            "--device",
            TRAIN_DEVICE,
            "--amp",
            AMP_MODE,
        ],
        check=True,
    )
    neural_best_params = json.loads(Path("outputs/neural_sweep_top100/best_params.json").read_text(encoding="utf-8"))
else:
    neural_best_params = {}

FINAL_NEURAL_MODEL = str(neural_best_params.get("model", "lstm"))
FINAL_NEURAL_RUN = f"colab_{FINAL_NEURAL_MODEL}_top100"
NEURAL_BATCH_SIZE = int(neural_best_params.get("batch_size", 32))
NEURAL_GRAD_ACCUM_STEPS = int(neural_best_params.get("grad_accum_steps", GRAD_ACCUM_STEPS))
NEURAL_GRAD_CLIP_NORM = float(neural_best_params.get("grad_clip_norm", GRAD_CLIP_NORM))
NEURAL_LR = float(neural_best_params.get("lr", 0.001))
NEURAL_WEIGHT_DECAY = float(neural_best_params.get("weight_decay", 0.0001))
SUMMARY_HIDDEN_DIM = int(neural_best_params.get("summary_hidden_dim", 128))
SUMMARY_DROPOUT = float(neural_best_params.get("summary_dropout", 0.2))
LSTM_EMBED_DIM = int(neural_best_params.get("lstm_embed_dim", 64))
LSTM_HIDDEN_DIM = int(neural_best_params.get("lstm_hidden_dim", 128))
LSTM_LAYERS = int(neural_best_params.get("lstm_layers", 2))
LSTM_DROPOUT = float(neural_best_params.get("lstm_dropout", 0.2))
LSTM_HEAD_DROPOUT = float(neural_best_params.get("lstm_head_dropout", 0.3))
TRANSFORMER_EMBED_DIM = int(neural_best_params.get("transformer_embed_dim", 64))
TRANSFORMER_HEADS = int(neural_best_params.get("transformer_heads", 4))
TRANSFORMER_LAYERS = int(neural_best_params.get("transformer_layers", 3))
TRANSFORMER_FF_DIM = int(neural_best_params.get("transformer_ff_dim", 256))
TRANSFORMER_DROPOUT = float(neural_best_params.get("transformer_dropout", 0.1))
TRANSFORMER_HEAD_DROPOUT = float(neural_best_params.get("transformer_head_dropout", 0.2))
neural_best_params


In [ ]:
import subprocess

cmd = [
    "python", "-m", "mania_difficulty.train",
    "--labels", "data/processed/labels.csv",
    "--sequences", "data/processed/sequences",
    "--epochs", "50",
    "--batch-size", str(NEURAL_BATCH_SIZE),
    "--grad-accum-steps", str(NEURAL_GRAD_ACCUM_STEPS),
    "--grad-clip-norm", str(NEURAL_GRAD_CLIP_NORM),
    "--checkpoint-metric", CHECKPOINT_METRIC,
    "--sample-weight-column", SAMPLE_WEIGHT_COLUMN,
    "--sample-weight-min", str(SAMPLE_WEIGHT_MIN),
    "--sample-weight-max-value", str(SAMPLE_WEIGHT_MAX_VALUE),
    "--run-name", FINAL_NEURAL_RUN,
    "--model", FINAL_NEURAL_MODEL,
    "--lr", str(NEURAL_LR),
    "--weight-decay", str(NEURAL_WEIGHT_DECAY),
    "--group-column", "beatmapset_id",
    "--loader-workers", str(LOADER_WORKERS),
    "--device", TRAIN_DEVICE,
    "--amp", AMP_MODE,
]
if RESUME_FINAL_TRAINING:
    cmd += ["--resume"]
if USE_DRIVE_CHECKPOINT_BACKUP:
    cmd += ["--checkpoint-backup-dir", CHECKPOINT_BACKUP_DIR]
if FINAL_NEURAL_MODEL == "summary":
    cmd += [
        "--summary-hidden-dim", str(SUMMARY_HIDDEN_DIM),
        "--summary-dropout", str(SUMMARY_DROPOUT),
    ]
elif FINAL_NEURAL_MODEL == "lstm":
    cmd += [
        "--lstm-embed-dim", str(LSTM_EMBED_DIM),
        "--lstm-hidden-dim", str(LSTM_HIDDEN_DIM),
        "--lstm-layers", str(LSTM_LAYERS),
        "--lstm-dropout", str(LSTM_DROPOUT),
        "--lstm-head-dropout", str(LSTM_HEAD_DROPOUT),
    ]
elif FINAL_NEURAL_MODEL == "transformer":
    cmd += [
        "--transformer-embed-dim", str(TRANSFORMER_EMBED_DIM),
        "--transformer-heads", str(TRANSFORMER_HEADS),
        "--transformer-layers", str(TRANSFORMER_LAYERS),
        "--transformer-ff-dim", str(TRANSFORMER_FF_DIM),
        "--transformer-dropout", str(TRANSFORMER_DROPOUT),
        "--transformer-head-dropout", str(TRANSFORMER_HEAD_DROPOUT),
    ]
subprocess.run(cmd, check=True)


In [ ]:
from google.colab import files
import subprocess
import pandas as pd
import torch
from pathlib import Path
from IPython.display import HTML, display
from mania_difficulty.tools.package_artifacts import package_artifacts

run_names = ["colab_forest_top100", FINAL_NEURAL_RUN]
run_dirs = [Path("outputs/runs") / run_name for run_name in run_names]
run_dirs = [run_dir for run_dir in run_dirs if run_dir.exists()]

for run_dir in run_dirs:
    checkpoint = run_dir / ("best_model.joblib" if (run_dir / "best_model.joblib").exists() else "best_model.pt")
    if checkpoint.exists():
        cmd = [
            "python", "-m", "mania_difficulty.tools.project_embeddings",
            "--checkpoint", str(checkpoint),
            "--labels", "data/processed/labels.csv",
            "--sequences", "data/processed/sequences",
            "--method", "pca",
            "--color-target", "mean_acc",
        ]
        if checkpoint.suffix == ".pt":
            cmd += ["--batch-size", "64"]
            if TRAIN_DEVICE == "cuda":
                cmd += ["--device", TRAIN_DEVICE]
        subprocess.run(cmd, check=True)
    if run_dir.name.startswith("colab_transformer_"):
        predictions_path = run_dir / "predictions.csv"
        if predictions_path.exists():
            attention_beatmap_id = int(pd.read_csv(predictions_path).iloc[0]["beatmap_id"])
            attention_cmd = [
                "python", "-m", "mania_difficulty.tools.attention_map",
                "--checkpoint", str(checkpoint),
                "--beatmap-id", str(attention_beatmap_id),
                "--sequences", "data/processed/sequences",
            ]
            if TRAIN_DEVICE == "cuda":
                attention_cmd += ["--device", TRAIN_DEVICE]
            subprocess.run(attention_cmd, check=True)

run_args = [str(run_dir) for run_dir in run_dirs]
subprocess.run(["python", "-m", "mania_difficulty.tools.compare_runs", *run_args, "--out-csv", "outputs/colab_top100_comparison.csv", "--out-html", "outputs/colab_top100_comparison.html"], check=True)
subprocess.run([
    "python", "-m", "mania_difficulty.tools.build_dashboard", *run_args,
    "--audit-dir", "outputs/dataset_audit_top100",
    "--forest-sweep-dir", "outputs/forest_sweep_top100",
    "--neural-sweep-dir", "outputs/neural_sweep_top100",
    "--comparison-html", "outputs/colab_top100_comparison.html",
    "--out-html", "outputs/colab_dashboard.html",
    "--decision-summary-csv", "outputs/colab_run_decision_summary.csv",
], check=True)
display(HTML(Path("outputs/colab_dashboard.html").read_text(encoding="utf-8")))
decision_summary = pd.read_csv("outputs/colab_run_decision_summary.csv")
display(decision_summary)
calibration_columns = [
    "run",
    "evaluation",
    "calibration_mean_abs_bias",
    "calibration_worst_bias_target",
    "calibration_worst_bias",
    "calibration_worst_p90_target",
    "calibration_worst_p90_abs_error",
    "calibration_mean_pred_std_ratio",
    "calibration_warning",
    "training_adjustment",
]
available_calibration_columns = [column for column in calibration_columns if column in decision_summary.columns]
if available_calibration_columns:
    print("Calibration signals from prediction_summary.csv / cv_prediction_summary.csv")
    display(decision_summary[available_calibration_columns])
artifact_paths = [
    "outputs/dataset_audit_top100",
    "outputs/forest_sweep_top100",
    *run_args,
    "outputs/colab_top100_comparison.csv",
    "outputs/colab_top100_comparison.html",
    "outputs/colab_run_decision_summary.csv",
    "outputs/colab_dashboard.html",
]
if Path("outputs/neural_sweep_top100").exists():
    artifact_paths.insert(1, "outputs/neural_sweep_top100")
manifest = package_artifacts(
    [Path(path) for path in artifact_paths],
    Path("colab_top100_outputs.zip"),
    manifest_path=Path("outputs/colab_artifact_manifest.json"),
    root=Path("."),
)
display(pd.DataFrame([manifest]))
files.download("colab_top100_outputs.zip")


## Inspect Real Runs

Use this to compare metrics, human-review candidates, plots, and HTML reports after training.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import FileLink, HTML, Image, display
from mania_difficulty.human_judgments import score_pair_judgments

audit_dir = Path("outputs/dataset_audit_top100")
if audit_dir.exists():
    print("Dataset audit")
    display(json.loads((audit_dir / "dataset_audit.json").read_text(encoding="utf-8")))
    display(Image(str(audit_dir / "dataset_distributions.png")))
    display(HTML((audit_dir / "dataset_audit.html").read_text(encoding="utf-8")))

forest_sweep_dir = Path("outputs/forest_sweep_top100")
if forest_sweep_dir.exists():
    print("Forest sweep best params")
    display(json.loads((forest_sweep_dir / "best_params.json").read_text(encoding="utf-8")))
    display(pd.read_csv(forest_sweep_dir / "sweep_summary.csv"))
    display(HTML((forest_sweep_dir / "sweep_report.html").read_text(encoding="utf-8")))

neural_sweep_dir = Path("outputs/neural_sweep_top100")
if neural_sweep_dir.exists():
    print("Neural sweep best params")
    display(json.loads((neural_sweep_dir / "best_params.json").read_text(encoding="utf-8")))
    display(pd.read_csv(neural_sweep_dir / "neural_sweep_summary.csv"))
    display(HTML((neural_sweep_dir / "neural_sweep_report.html").read_text(encoding="utf-8")))

dashboard_path = Path("outputs/colab_dashboard.html")
if dashboard_path.exists():
    print("Dashboard")
    display(HTML(dashboard_path.read_text(encoding="utf-8")))

run_names = ["colab_forest_top100", str(globals().get("FINAL_NEURAL_RUN", "colab_lstm_top100"))]
metric_rows = []
run_rows = []
for run_name in run_names:
    run_dir = Path("outputs/runs") / run_name
    for metrics_name, evaluation in [("metrics.json", "holdout"), ("cv_metrics.json", "cv_oof")]:
        metrics_path = run_dir / metrics_name
        if not metrics_path.exists():
            continue
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        for target, values in metrics.items():
            if isinstance(values, dict) and "mae" in values:
                metric_rows.append({"run": run_name, "evaluation": evaluation, "target": target, **values})
        run_rows.append({"run": run_name, "evaluation": evaluation, **metrics["_run"]})

metrics_frame = pd.DataFrame(metric_rows).sort_values(["evaluation", "target", "mae"])
display(metrics_frame)
display(pd.DataFrame(run_rows))

for run_name in run_names:
    run_dir = Path("outputs/runs") / run_name
    print("\nRUN", run_name)
    history = pd.read_csv(run_dir / "history.csv")
    review_path = run_dir / "cv_human_review.csv"
    if not review_path.exists():
        review_path = run_dir / "human_review.csv"
    pair_review_path = run_dir / "cv_human_pair_review.csv"
    if not pair_review_path.exists():
        pair_review_path = run_dir / "human_pair_review.csv"
    judgment_template_path = run_dir / "cv_human_pair_judgment_template.csv"
    if not judgment_template_path.exists():
        judgment_template_path = run_dir / "human_pair_judgment_template.csv"
    error_slices_path = run_dir / "cv_error_slices.csv"
    if not error_slices_path.exists():
        error_slices_path = run_dir / "error_slices.csv"
    prediction_summary_path = run_dir / "cv_prediction_summary.csv"
    if not prediction_summary_path.exists():
        prediction_summary_path = run_dir / "prediction_summary.csv"
    human_review = pd.read_csv(review_path)
    print("Last 10 training epochs")
    display(history.tail(10))
    if prediction_summary_path.exists():
        print("Prediction calibration", prediction_summary_path.name)
        display(pd.read_csv(prediction_summary_path))
    print("Rows to human-review first", review_path.name)
    display(human_review.head(20))
    if pair_review_path.exists():
        print("Pairwise disagreements to compare by hand", pair_review_path.name)
        display(pd.read_csv(pair_review_path).head(20))
    if judgment_template_path.exists():
        print("Fill this CSV to score model agreement with your judgment", judgment_template_path.name)
        display(pd.read_csv(judgment_template_path).head(20))
        print("Human judgment score")
        display(pd.DataFrame([score_pair_judgments(judgment_template_path)]))
    if error_slices_path.exists():
        print("Error slices by metadata", error_slices_path.name)
        display(pd.read_csv(error_slices_path).head(20))
    embedding_report_path = run_dir / "embedding_report.html"
    if embedding_report_path.exists():
        print("Embedding projection")
        display(HTML(embedding_report_path.read_text(encoding="utf-8")))
    attention_report_path = run_dir / "attention_report.html"
    if attention_report_path.exists():
        print("Transformer attention map")
        display(HTML(attention_report_path.read_text(encoding="utf-8")))
    display(Image(str(run_dir / "prediction_scatter.png")))
    prediction_errors = run_dir / "prediction_errors.png"
    if prediction_errors.exists():
        display(Image(str(prediction_errors)))
    cv_scatter = run_dir / "cv_prediction_scatter.png"
    if cv_scatter.exists():
        display(Image(str(cv_scatter)))
    cv_errors = run_dir / "cv_prediction_errors.png"
    if cv_errors.exists():
        display(Image(str(cv_errors)))
    display(Image(str(run_dir / "learning_curve.png")))
    feature_plot = run_dir / "feature_importance.png"
    if feature_plot.exists():
        display(Image(str(feature_plot)))
    display(HTML((run_dir / "run_report.html").read_text(encoding="utf-8")))

artifact_manifest_path = Path("outputs/colab_artifact_manifest.json")
if artifact_manifest_path.exists():
    print("Artifact manifest")
    display(json.loads(artifact_manifest_path.read_text(encoding="utf-8")))
    display(FileLink(str(artifact_manifest_path)))
display(FileLink("colab_top100_outputs.zip"))
